# 01 Ingestion And Table Anatomy

Inspect the table inventory, parse statuses, metadata/time boundary, and warning cases used by later assignment steps.


In [1]:
from pathlib import Path
import csv
import json

import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


def read_json(relative_path: str):
    path = ROOT / relative_path
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"missing": str(path)}


def csv_rows(relative_path: str, limit: int | None = None):
    csv.field_size_limit(2_147_483_647)
    path = ROOT / relative_path
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = list(csv.DictReader(file))
    return rows if limit is None else rows[:limit]


def csv_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    if not path.exists():
        return None
    return len(csv_rows(relative_path))


def parquet_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    return pq.read_table(path).num_rows if path.exists() else None


## Parse Summary


In [2]:
diagnostics = read_json("data/processed/ingestion_diagnostics.json")
{
    "inventoried_table_count": diagnostics.get("inventoried_table_count"),
    "status_counts": diagnostics.get("status_counts"),
    "failed_table_count": diagnostics.get("failed_table_count"),
    "warning_table_count": len(diagnostics.get("warning_tables", [])),
}


{'inventoried_table_count': 2000,
 'status_counts': {'parsed': 1771, 'warning': 229},
 'failed_table_count': 0,
 'warning_table_count': 229}

## Table Anatomy Examples


In [3]:
path = ROOT / "data/processed/tables.parquet"
if path.exists():
    rows = pq.read_table(path).slice(0, 5).to_pylist()
    [
        {
            "table_id": row.get("table_id"),
            "title": row.get("title"),
            "parse_status": row.get("parse_status"),
            "metadata_column_count": row.get("metadata_column_count"),
            "time_column_count": row.get("time_column_count"),
        }
        for row in rows
    ]
else:
    {"missing": str(path)}


## Warning Examples


In [4]:
diagnostics.get("warning_tables", [])[:10]


[{'parse_status': 'warning',
  'reasons': ['missing title'],
  'table_id': 'bd_size$dv_1502'},
 {'parse_status': 'warning',
  'reasons': ['missing title'],
  'table_id': 'crim_off_cat$dv_1401'},
 {'parse_status': 'warning',
  'reasons': ['missing title'],
  'table_id': 'crim_off_cat$dv_348'},
 {'parse_status': 'warning',
  'reasons': ['missing title'],
  'table_id': 'crim_thb_sex$dv_1641'},
 {'parse_status': 'warning',
  'reasons': ['missing title'],
  'table_id': 'crim_thb_vctz$dv_1643'},
 {'parse_status': 'warning',
  'reasons': ['missing title'],
  'table_id': 'demo_magec$dv_1942'},
 {'parse_status': 'warning',
  'reasons': ['missing title'],
  'table_id': 'demo_mlexpec$dv_292'},
 {'parse_status': 'warning',
  'reasons': ['missing title'],
  'table_id': 'demo_r_find3$dv_1541'},
 {'parse_status': 'warning',
  'reasons': ['missing title'],
  'table_id': 'demo_r_gind3$dv_1542'},
 {'parse_status': 'warning',
  'reasons': ['missing title'],
  'table_id': 'demo_r_pjanind3$dv_1561'}]